In [1]:
from pathlib import Path
import pandas as pd
import openpyxl

def locate_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    return current

root = locate_root()
data_dir = root / "data"
metadata_dir = data_dir / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)
xlsx_files = sorted([p for p in data_dir.rglob("*.xlsx") if p.is_file()])
len(xlsx_files)

37

In [2]:
def classify_group(path):
    folder = path.parent.name.lower()
    name = path.name.lower()
    if "healthy" in folder or "healthy" in name:
        return "healthy"
    if "defected" in folder or "defective" in folder or "defect" in name:
        return "defective"
    return "unassigned"

inventory = pd.DataFrame(
    {
        "file_name": [p.name for p in xlsx_files],
        "relative_path": [str(p.relative_to(root)) for p in xlsx_files],
        "folder": [p.parent.name for p in xlsx_files],
        "suffix": [p.suffix.lower() for p in xlsx_files],
        "size_bytes": [p.stat().st_size for p in xlsx_files],
        "group": [classify_group(p) for p in xlsx_files],
    }
)
inventory

,file_name,relative_path,folder,suffix,size_bytes,group
0,Defect15(01).xlsx,data/raw/defective/Defect15(01).xlsx,defective,.xlsx,63799916,defective
1,Defect15(02).xlsx,data/raw/defective/Defect15(02).xlsx,defective,.xlsx,6907967,defective
2,Defect15(03).xlsx,data/raw/defective/Defect15(03).xlsx,defective,.xlsx,77841167,defective
3,Defect15(04).xlsx,data/raw/defective/Defect15(04).xlsx,defective,.xlsx,40512363,defective
4,Defect15.xlsx,data/raw/defective/Defect15.xlsx,defective,.xlsx,7637,defective
5,Defect16(01).xlsx,data/raw/defective/Defect16(01).xlsx,defective,.xlsx,11509691,defective
6,Defect16(02).xlsx,data/raw/defective/Defect16(02).xlsx,defective,.xlsx,30648268,defective
7,Defect16(03).xlsx,data/raw/defective/Defect16(03).xlsx,defective,.xlsx,66812842,defective
8,Defected 1.xlsx,data/raw/defective/Defected 1.xlsx,defective,.xlsx,63801327,defective
9,Defected 2.xlsx,data/raw/defective/Defected 2.xlsx,defective,.xlsx,6908662,defective


In [3]:
def workbook_profile(path):
    try:
        wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
        sheet_names = wb.sheetnames
        rows = []
        for sheet_name in sheet_names:
            ws = wb[sheet_name]
            rows.append(
                {
                    "file_name": path.name,
                    "sheet_name": sheet_name,
                    "max_row": ws.max_row,
                    "max_column": ws.max_column,
                }
            )
        wb.close()
        return rows
    except Exception as e:
        return [
            {
                "file_name": path.name,
                "sheet_name": None,
                "max_row": None,
                "max_column": None,
                "error": str(e),
            }
        ]

sheet_profiles = []
for path in xlsx_files:
    sheet_profiles.extend(workbook_profile(path))

sheet_inventory = pd.DataFrame(sheet_profiles)
sheet_inventory

,file_name,sheet_name,max_row,max_column
0,Defect15(01).xlsx,04270743,8180,209
1,Defect15(01).xlsx,06121032,11082,208
2,Defect15(01).xlsx,06122027,2929,207
3,Defect15(01).xlsx,08180820,12432,208
4,Defect15(01).xlsx,08181312,10391,208
...,...,...,...,...
147,SatFiles.xlsx,04192910,13113,208
148,SatFiles.xlsx,05271031,12633,208
149,SatFiles.xlsx,05110715,11503,208
150,SatFiles.xlsx,05130829,13120,208


In [4]:
inventory_summary = (
    inventory.groupby(["group", "folder"], dropna=False)
    .agg(files=("file_name", "nunique"), total_size_bytes=("size_bytes", "sum"))
    .reset_index()
    .sort_values(["group", "folder"])
)
inventory_summary

,group,folder,files,total_size_bytes
0,defective,defective,15,596079229
1,healthy,healthy,9,626706039
2,unassigned,reference,13,638365650


In [5]:
sheet_summary = (
    sheet_inventory.groupby("file_name", dropna=False)
    .agg(
        sheets=("sheet_name", "nunique"),
        total_rows=("max_row", "sum"),
        max_rows=("max_row", "max"),
        max_columns=("max_column", "max"),
    )
    .reset_index()
    .sort_values(["max_rows", "max_columns"], ascending=[False, False])
)
sheet_summary

,file_name,sheets,total_rows,max_rows,max_columns
35,Healthy 9.xlsx,13,176280,20463,209
36,SatFiles.xlsx,13,176280,20463,209
9,01012.xlsx,10,135179,19983,208
32,Healthy 6.xlsx,10,135179,19983,208
1,00101206.xlsx,1,19983,19983,13
4,01007.xlsx,5,62310,19621,208
27,Healthy 1.xlsx,5,62310,19621,208
3,003.xlsx,5,59585,19178,13
8,01011.xlsx,4,46561,19128,208
31,Healthy 5.xlsx,4,46561,19128,208


In [6]:
inventory.to_csv(metadata_dir / "file_inventory.csv", index=False)
sheet_inventory.to_csv(metadata_dir / "sheet_inventory.csv", index=False)
sheet_summary.to_csv(metadata_dir / "sheet_summary.csv", index=False)

{
    "inventory": str(metadata_dir / "file_inventory.csv"),
    "sheet_inventory": str(metadata_dir / "sheet_inventory.csv"),
    "sheet_summary": str(metadata_dir / "sheet_summary.csv"),
}

{'inventory': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/file_inventory.csv',
 'sheet_inventory': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/sheet_inventory.csv',
 'sheet_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/sheet_summary.csv'}